# AI-Based Violence Detection & Video Alert System

**Author:** Balaji  
**Environment:** Google Colab  
**Task:** Binary video classification — Violence/Fight vs Non-Violence/Non-Fight

This notebook builds a reproducible video-classification pipeline. It samples frames from each video, extracts visual embeddings with a pretrained MobileNetV2 backbone, learns temporal patterns with a GRU network, evaluates the model, and predicts on new videos.

> **Responsible-use note:** This is an experimental decision-support prototype, not a law-enforcement tool. It can produce false positives/negatives and must not be used as the sole basis for accusations, intervention, or disciplinary action. Use only lawfully obtained, consented datasets. Do not identify people.

## Pipeline
1. Configure and load a labeled video dataset.
2. Detect class folders and create stratified train/validation/test splits (or honor existing train/test folders).
3. Sample a fixed number of frames per clip.
4. Extract MobileNetV2 embeddings and cache them.
5. Train a GRU classifier with class weighting and early stopping.
6. Evaluate with accuracy, precision, recall, F1, ROC-AUC, confusion matrix and classification report.
7. Run inference on an uploaded video and export results.

### Expected dataset layout
The loader accepts either a split-based layout or a class-folder layout. Common class names such as `Fight`, `Violence`, `NonFight`, and `NonViolence` are recognized.

```text
dataset/
  train/
    Fight/       video files
    NonFight/    video files
  test/
    Fight/       video files
    NonFight/    video files
```

A single class-folder root is also accepted:

```text
dataset/
  Violence/
  NonViolence/
```

For RWF-2000-style data, extract the dataset ZIP into `/content/dataset` and point `DATASET_DIR` there. Dataset download is intentionally not automated because source access, licensing and credentials vary.

## 1. Install and import dependencies
Colab usually includes TensorFlow. The install cell adds video and evaluation utilities.

In [ ]:
!pip -q install opencv-python-headless scikit-learn seaborn pandas matplotlib

In [ ]:
import os, cv2, json, math, random, zipfile, shutil, warnings, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Configure the dataset
Upload a ZIP through the Colab file picker, or mount a dataset already available in the runtime. The notebook does not download third-party datasets automatically.

In [ ]:
from google.colab import files

# Option A: upload a dataset ZIP. It will be extracted to /content/dataset_upload.
UPLOAD_DATASET_ZIP = True
if UPLOAD_DATASET_ZIP:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if zip_names:
        extract_root = Path('/content/dataset_upload')
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_names[0], 'r') as zf:
            zf.extractall(extract_root)
        print('Extracted:', zip_names[0], 'to', extract_root)
    else:
        print('No ZIP selected. Set DATASET_DIR manually below if data is already present.')

# Change this to the actual folder containing train/test or class folders.
DATASET_DIR = '/content/dataset_upload'
# If the ZIP contains one wrapper folder, the discovery cell below finds the data root.
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg'}
CLASS_ALIASES = {
    1: {'fight','fighting','violence','violent','assault'},
    0: {'nonfight','non_fight','non- fight','nonviolence','non_violence','non-violence','normal','nofight','no_fight','nonviolent'}
}
IMG_SIZE = 160
NUM_FRAMES = 16
BATCH_SIZE = 32
EPOCHS = 20
AUTOTUNE = tf.data.AUTOTUNE

## 3. Discover labeled videos
The class-folder names determine labels. The code supports folder names `Fight`/`NonFight`, `Violence`/`NonViolence`, and common variants. It checks for split folders named `train`, `val`/`validation`, and `test`.

In [ ]:
def normalize_name(name):
    return name.lower().replace(' ', '').replace('-', '').replace('_', '')

def label_from_folder(folder_name):
    key = normalize_name(folder_name)
    positive = {normalize_name(x) for x in CLASS_ALIASES[1]}
    negative = {normalize_name(x) for x in CLASS_ALIASES[0]}
    if key in positive: return 1
    if key in negative: return 0
    return None

def find_dataset_root(base):
    base = Path(base)
    if not base.exists(): raise FileNotFoundError(f'Dataset directory not found: {base}')
    candidates = [base] + [p for p in base.iterdir() if p.is_dir()]
    for candidate in candidates:
        names = {normalize_name(p.name) for p in candidate.rglob('*') if p.is_dir()}
        has_pos = any(n in {normalize_name(x) for x in CLASS_ALIASES[1]} for n in names)
        has_neg = any(n in {normalize_name(x) for x in CLASS_ALIASES[0]} for n in names)
        if has_pos and has_neg: return candidate
    return base

DATA_ROOT = find_dataset_root(DATASET_DIR)
print('Candidate data root:', DATA_ROOT)

def collect_split(split_name):
    split_dir = next((p for p in DATA_ROOT.iterdir() if p.is_dir() and normalize_name(p.name)==normalize_name(split_name)), None)
    rows=[]
    if split_dir is None: return rows
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir(): continue
        label=label_from_folder(class_dir.name)
        if label is None: continue
        for f in class_dir.rglob('*'):
            if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS: rows.append((str(f),label))
    return rows

train_rows = collect_split('train')
val_rows = collect_split('val') or collect_split('validation')
test_rows = collect_split('test')
if not train_rows and not test_rows:
    # Search class directories recursively when there are no explicit split folders.
    all_rows=[]
    for folder in DATA_ROOT.rglob('*'):
        if folder.is_dir():
            label=label_from_folder(folder.name)
            if label is not None:
                for f in folder.rglob('*'):
                    if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS: all_rows.append((str(f),label))
    # Deduplicate paths while preserving order.
    all_rows=list({p:(p,y) for p,y in all_rows}.values())
    if len(all_rows)<10: raise ValueError(f'Only {len(all_rows)} labeled videos found. Check DATASET_DIR and class folder names.')
    paths=np.array([r[0] for r in all_rows]); labels=np.array([r[1] for r in all_rows])
    tr_p, temp_p, tr_y, temp_y = train_test_split(paths, labels, test_size=0.30, random_state=SEED, stratify=labels)
    va_p, te_p, va_y, te_y = train_test_split(temp_p, temp_y, test_size=0.50, random_state=SEED, stratify=temp_y)
    train_rows=list(zip(tr_p,tr_y)); val_rows=list(zip(va_p,va_y)); test_rows=list(zip(te_p,te_y))
elif not val_rows:
    # Create validation from train only; preserve explicit test folder.
    p=np.array([r[0] for r in train_rows]); y=np.array([r[1] for r in train_rows])
    tr_p, va_p, tr_y, va_y=train_test_split(p,y,test_size=0.20,random_state=SEED,stratify=y)
    train_rows=list(zip(tr_p,tr_y)); val_rows=list(zip(va_p,va_y))

for split_name, rows in [('train',train_rows),('validation',val_rows),('test',test_rows)]:
    counts=pd.Series([y for _,y in rows]).value_counts().to_dict()
    print(f'{split_name:10s}: {len(rows):5d} videos | label counts {counts}')
if not train_rows or not val_rows or not test_rows:
    raise ValueError('Train/validation/test split is empty. Supply enough videos in both classes and check folder layout.')
if len(set(y for _,y in train_rows))<2: raise ValueError('Training data must contain both classes.')

## 4. Sample frames from each video
Frames are sampled at evenly spaced positions, resized to 160×160, and padded by repeating the last valid frame if a clip is short. Corrupt or unreadable clips are skipped and reported.

In [ ]:
def read_video_frames(video_path, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
    cap=cv2.VideoCapture(video_path)
    if not cap.isOpened(): return None
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames=[]
    if total>0:
        indices=np.linspace(0,max(total-1,0),num_frames).astype(int)
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx))
            ok, frame=cap.read()
            if ok:
                frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                frame=cv2.resize(frame,(img_size,img_size))
                frames.append(frame)
    else:
        while len(frames)<num_frames:
            ok,frame=cap.read()
            if not ok: break
            frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            frame=cv2.resize(frame,(img_size,img_size))
            frames.append(frame)
    cap.release()
    if not frames: return None
    while len(frames)<num_frames: frames.append(frames[-1].copy())
    if len(frames)>num_frames: frames=frames[:num_frames]
    return np.asarray(frames,dtype=np.uint8)

# Preview one clip before processing the full dataset.
preview_frames=read_video_frames(train_rows[0][0])
if preview_frames is None: raise RuntimeError('Could not decode the first training video. Check video codecs/files.')
plt.figure(figsize=(12,3))
for i in range(min(4,NUM_FRAMES)):
    plt.subplot(1,4,i+1); plt.imshow(preview_frames[i]); plt.axis('off'); plt.title(f'Frame {i+1}')
plt.tight_layout(); plt.show()

## 5. Extract pretrained visual embeddings
MobileNetV2 is used as a frozen feature extractor. The resulting sequence of frame embeddings is much smaller than storing full video frames and is then passed to a GRU for temporal learning. Embeddings are cached in the Colab runtime.

In [ ]:
feature_extractor=MobileNetV2(include_top=False,weights='imagenet',pooling='avg',input_shape=(IMG_SIZE,IMG_SIZE,3))
feature_extractor.trainable=False
print('Feature dimension:', feature_extractor.output_shape[-1])

def video_to_embedding(video_path):
    frames=read_video_frames(video_path)
    if frames is None: return None
    x=preprocess_input(frames.astype(np.float32))
    feats=feature_extractor.predict(x,batch_size=NUM_FRAMES,verbose=0)
    return feats.astype(np.float32)

def build_feature_set(rows, split_label):
    X=[]; y=[]; kept=[]; failed=[]
    for i,(path,label) in enumerate(rows,1):
        feat=video_to_embedding(path)
        if feat is None:
            failed.append(path); continue
        X.append(feat); y.append(int(label)); kept.append(path)
        if i%25==0: print(f'{split_label}: processed {i}/{len(rows)}')
    if not X: raise RuntimeError(f'No readable videos in {split_label}.')
    return np.stack(X),np.asarray(y,dtype=np.int32),kept,failed

X_train,y_train,train_paths,train_failed=build_feature_set(train_rows,'train')
X_val,y_val,val_paths,val_failed=build_feature_set(val_rows,'validation')
X_test,y_test,test_paths,test_failed=build_feature_set(test_rows,'test')
print('Feature shapes:',X_train.shape,X_val.shape,X_test.shape)
print('Unreadable videos:',len(train_failed)+len(val_failed)+len(test_failed))

## 6. Build and train the temporal classifier
The GRU learns temporal changes across the frame-embedding sequence. Class weights help when the classes are imbalanced. The held-out test split is not used for training or model selection.

In [ ]:
num_classes=len(np.unique(y_train))
weights=compute_class_weight(class_weight='balanced',classes=np.unique(y_train),y=y_train)
class_weight={int(c):float(w) for c,w in zip(np.unique(y_train),weights)}

def make_model():
    inputs=keras.Input(shape=(NUM_FRAMES, X_train.shape[-1]),name='frame_embeddings')
    x=layers.GRU(128,return_sequences=True,dropout=0.25)(inputs)
    x=layers.GRU(64,dropout=0.25)(x)
    x=layers.Dense(64,activation='relu')(x)
    x=layers.Dropout(0.35)(x)
    outputs=layers.Dense(1,activation='sigmoid',name='violence_probability')(x)
    model=keras.Model(inputs,outputs,name='mobilenet_gru_violence_detector')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',metrics=['accuracy',keras.metrics.Precision(name='precision'),keras.metrics.Recall(name='recall'),keras.metrics.AUC(name='auc')])
    return model

model=make_model()
model.summary()
callbacks=[
    keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2,min_lr=1e-6),
]
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=EPOCHS,batch_size=BATCH_SIZE,
    class_weight=class_weight,callbacks=callbacks,verbose=1)

## 7. Training curves

In [ ]:
hist=pd.DataFrame(history.history)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(hist['loss'],label='Train'); axes[0].plot(hist['val_loss'],label='Validation'); axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(hist['accuracy'],label='Train'); axes[1].plot(hist['val_accuracy'],label='Validation'); axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Evaluate on the held-out test set
The positive class is Violence/Fight (label 1). ROC-AUC is reported only when both classes are present in the test labels.

In [ ]:
test_prob=model.predict(X_test,batch_size=BATCH_SIZE,verbose=0).ravel()
THRESHOLD=0.50
test_pred=(test_prob>=THRESHOLD).astype(int)
metrics={
    'accuracy':float(accuracy_score(y_test,test_pred)),
    'precision':float(precision_score(y_test,test_pred,zero_division=0)),
    'recall':float(recall_score(y_test,test_pred,zero_division=0)),
    'f1':float(f1_score(y_test,test_pred,zero_division=0)),
    'roc_auc':float(roc_auc_score(y_test,test_prob)) if len(np.unique(y_test))==2 else None,
    'threshold':THRESHOLD, 'test_videos':int(len(y_test))
}
print(json.dumps(metrics,indent=2))
print('\nClassification report:\n',classification_report(y_test,test_pred,target_names=['Non-Violence','Violence'],zero_division=0))
cm=confusion_matrix(y_test,test_pred,labels=[0,1])
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Non-Violence','Violence'],yticklabels=['Non-Violence','Violence'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Test Confusion Matrix'); plt.show()
if metrics['roc_auc'] is not None:
    fpr,tpr,_=roc_curve(y_test,test_prob); plt.figure(figsize=(5,4)); plt.plot(fpr,tpr,label=f"AUC={metrics['roc_auc']:.3f}"); plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.show()

## 9. Save model, metrics and test predictions
The saved classifier expects MobileNetV2 embeddings, so save both the GRU model and configuration needed to reproduce feature extraction.

In [ ]:
OUTPUT_DIR=Path('/content/violence_detection_outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
model.save(OUTPUT_DIR/'violence_gru_classifier.keras')
with open(OUTPUT_DIR/'metrics.json','w') as f: json.dump(metrics,f,indent=2)
config={'img_size':IMG_SIZE,'num_frames':NUM_FRAMES,'feature_backbone':'MobileNetV2','embedding_dim':int(X_train.shape[-1]),'threshold':THRESHOLD,'class_mapping':{'0':'Non-Violence','1':'Violence'}}
with open(OUTPUT_DIR/'config.json','w') as f: json.dump(config,f,indent=2)
pred_df=pd.DataFrame({'video_path':test_paths,'actual_label':y_test,'predicted_label':test_pred,'violence_probability':test_prob})
pred_df.to_csv(OUTPUT_DIR/'test_predictions.csv',index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR/'evaluation_metrics.csv',index=False)
print('Saved files:',[p.name for p in OUTPUT_DIR.iterdir()])

## 10. Predict on a new video
Upload a single video file. The model returns a probability and a threshold-based label. A high probability is not proof that violence occurred.

In [ ]:
uploaded_video=files.upload()
video_candidates=[n for n in uploaded_video if Path(n).suffix.lower() in VIDEO_EXTENSIONS]
if video_candidates:
    new_video=video_candidates[0]
    new_embedding=video_to_embedding(new_video)
    if new_embedding is None:
        print('Could not decode video:',new_video)
    else:
        probability=float(model.predict(new_embedding[np.newaxis,...],verbose=0)[0][0])
        prediction='Violence/Fight' if probability>=THRESHOLD else 'Non-Violence/Non-Fight'
        print({'video':new_video,'prediction':prediction,'violence_probability':round(probability,4),'threshold':THRESHOLD})
else:
    print('No supported video file was uploaded.')

## 11. Optional: export a short event-style frame summary
This is a clip-level model, not a frame-level localization model. The chart below visualizes the predicted clip score; it does not indicate the exact time or location of an event.

In [ ]:
# Example visual summary for the uploaded clip if it was successfully scored above.
if 'probability' in globals():
    plt.figure(figsize=(5,2.5)); plt.bar(['Violence probability'],[probability],color='tomato'); plt.ylim(0,1); plt.ylabel('Probability'); plt.title('Clip-level model score'); plt.show()

## 12. Limitations, ethics and reproducibility
- Performance depends on dataset quality, camera angles, lighting, compression, scene context and class balance.
- Dataset-level splits must avoid near-duplicate clips and, where possible, separate camera/source identities to reduce leakage.
- A clip-level binary label does not localize the event or explain why the model made a prediction.
- MobileNetV2 embeddings are frozen; domain-specific fine-tuning may improve performance but requires careful validation.
- The threshold of 0.50 is a default, not a universal operating point. Choose thresholds using validation data and the costs of false positives/false negatives.
- Do not deploy for automated punishment, surveillance decisions, or emergency dispatch without independent validation, human review, privacy safeguards and legal approval.

### Suggested future improvements
1. Evaluate on a separate camera/source dataset.
2. Compare GRU, LSTM and temporal convolution baselines.
3. Tune the decision threshold on validation data.
4. Add calibration and per-camera error analysis.
5. Consider lightweight edge deployment only after external validation.

## 13. Download experiment outputs

In [ ]:
from google.colab import files
zip_path='/content/violence_detection_outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file(): zf.write(p,arcname=p.relative_to(OUTPUT_DIR))
files.download(zip_path)